# Mapping attributions

`map_node_attributions()` attaches `GraphSpec` node names to a
tensor of per-unit scores for **one** layer. The result is an
`xarray.DataArray`. Values are copied with `detach()` onto CPU.

The function does **not** run an attribution method, import
Captum, or sum or average scores. You compute the tensor (Captum
`LayerConductance`, input gradients, or any other scores with the
same width as that layer) and pass it in.

This notebook uses **fixed fake tensors** so the mapping is
visible without training. For a trained model plus Captum, see
[Getting started](../getting-started/). The signature is on the
[API page](../api/#map_node_attributions).


## Layers are GraphSpec depths

Captum scores **modules**. `GraphSpec` scores **layers**.
`layer=` is an index into `spec.layer_nodes`, not a Captum
target name.

- Layer `0` is the input nodes.
- A `MaskedLinear` built from `spec.masks[i]` writes the
  activations at layer `i + 1`. Pass `layer=i + 1` for that
  hop's output scores.
- Do **not** name-map BatchNorm or other unnamed modules. Only
  map tensors whose units are GraphSpec nodes.

Two layers can share a width. The function does not guess the
layer from `tensor.shape[-1]`.


## A tiny layered graph

Inputs `A` and `B` feed hidden nodes `H1` and `H2`, which feed
output `C`. Layers 0 and 1 both have width 2; layer 2 has width
1. That is the case that makes `layer=` mandatory.


In [1]:
import pandas as pd
import torch

import kpnn2 as k2

edgelist = pd.DataFrame(
    {
        "source": ["A", "B", "A", "B", "H1", "H2"],
        "target": ["H1", "H1", "H2", "H2", "C", "C"],
    }
)
spec = k2.parse_edgelist(edgelist)

print(
    "layer_nodes:",
    spec.layer_nodes,
)
print(
    "layer_dims:",
    spec.layer_dims,
)
print(
    "n masks:",
    len(spec.masks),
)

layer_nodes: (('A', 'B'), ('H1', 'H2'), ('C',))
layer_dims: (2, 2, 1)
n masks: 2


`spec.masks[i]` is the adjacent hop from layer `i` to layer
`i + 1`. The hop output has the same width as
`spec.layer_nodes[i + 1]`.


In [2]:
hop0 = k2.MaskedLinear(spec.masks[0])
hop1 = k2.MaskedLinear(spec.masks[1])
print(
    "masks[0] out_features:",
    hop0.out_features,
)
print(
    "layer 1 width:",
    spec.layer_dims[1],
)
print(
    "masks[1] out_features:",
    hop1.out_features,
)
print(
    "layer 2 width:",
    spec.layer_dims[2],
)

masks[0] out_features: 2
layer 1 width: 2
masks[1] out_features: 1
layer 2 width: 1


## Same width, two meanings

A `(2, 2)` tensor is a legal score matrix at layer 0 **and** at
layer 1. The names come only from `layer=`.


In [3]:
scores_2x2 = torch.tensor(
    [
        [0.10, 0.20],
        [0.30, 0.40],
    ]
)

named_inputs = k2.map_node_attributions(
    attributions=scores_2x2,
    spec=spec,
    layer=0,
)
named_hidden = k2.map_node_attributions(
    attributions=scores_2x2,
    spec=spec,
    layer=1,
)

print(
    "layer 0 nodes:",
    named_inputs["node"].values.tolist(),
)
print(
    "layer 1 nodes:",
    named_hidden["node"].values.tolist(),
)
print(
    "scalar layer coord (inputs):",
    int(named_inputs.coords["layer"]),
)
print(
    "scalar layer coord (hidden):",
    int(named_hidden.coords["layer"]),
)

layer 0 nodes: ['A', 'B']
layer 1 nodes: ['H1', 'H2']
scalar layer coord (inputs): 0
scalar layer coord (hidden): 1


## Default 2-D layout

A 2-D tensor is read as `(observation, node)`. Dim 1 must equal
`len(spec.layer_nodes[layer])`. The DataArray keeps the raw
values; nothing is reduced.


In [4]:
hidden = k2.map_node_attributions(
    attributions=scores_2x2,
    spec=spec,
    layer=1,
)
print(
    "dims:",
    hidden.dims,
)
print(hidden)

dims: ('observation', 'node')
<xarray.DataArray (observation: 2, node: 2)> Size: 16B
array([[0.1, 0.2],
       [0.3, 0.4]], dtype=float32)
Coordinates:
  * observation  (observation) int64 16B 0 1
  * node         (node) <U2 16B 'H1' 'H2'
    layer        int64 8B 1


## Long table vs wide table

xarray can flatten to pandas in two ways. A **long** table has
one score per row (good for plotting). A **wide** table has
nodes as columns (good for a quick look at one layer).


In [5]:
long_table = hidden.to_dataframe(name="score").reset_index()
wide_table = hidden.to_pandas()

print("long table:")
print(long_table)
print()
print("wide table:")
print(wide_table)

long table:
   observation node  layer  score
0            0   H1      1    0.1
1            0   H2      1    0.2
2            1   H1      1    0.3
3            1   H2      1    0.4

wide table:
node          H1   H2
observation          
0            0.1  0.2
1            0.3  0.4


## Select and reduce in xarray

Aggregation is **your** job. The mapper never sums or averages.
Use `sel` for named nodes and xarray reductions for extra axes.


In [6]:
print("H1 over observations:")
print(hidden.sel(node="H1"))
print()
print("mean over observation (still per node):")
print(hidden.mean("observation"))

H1 over observations:
<xarray.DataArray (observation: 2)> Size: 8B
array([0.1, 0.3], dtype=float32)
Coordinates:
  * observation  (observation) int64 16B 0 1
    node         <U2 8B 'H1'
    layer        int64 8B 1

mean over observation (still per node):
<xarray.DataArray (node: 2)> Size: 8B
array([0.2, 0.3], dtype=float32)
Coordinates:
  * node     (node) <U2 16B 'H1' 'H2'
    layer    int64 8B 1


## 1-D scores

A vector is labeled as dim `(node,)`. That is the layout for a
single observation, or for scores you already averaged yourself.


In [7]:
one_row = torch.tensor([0.90, 1.10])
vec = k2.map_node_attributions(
    attributions=one_row,
    spec=spec,
    layer=1,
)
print(
    "dims:",
    vec.dims,
)
print(vec.to_dataframe(name="score").reset_index())

dims: ('node',)
  node  layer  score
0   H1      1    0.9
1   H2      1    1.1


## Extra axes: pass `dims` and `coords`

Rank 3 or higher has no default names. `dims` must contain
`node` exactly once. `coords` may label the other axes; it must
**not** include `node` or `layer` (`layer` is always the scalar
you passed in).


In [8]:
# Shape (observation, class, node) at the hidden layer.
scores_cls = torch.tensor(
    [
        [[0.10, 0.20], [0.30, 0.40]],
        [[0.50, 0.60], [0.70, 0.80]],
    ]
)
by_class = k2.map_node_attributions(
    attributions=scores_cls,
    spec=spec,
    layer=1,
    dims=("observation", "class", "node"),
    coords={"class": ["neg", "pos"]},
)
print(
    "dims:",
    by_class.dims,
)
print(by_class.to_dataframe(name="score").reset_index())

dims: ('observation', 'class', 'node')
   observation class node  layer  score
0            0   neg   H1      1    0.1
1            0   neg   H2      1    0.2
2            0   pos   H1      1    0.3
3            0   pos   H2      1    0.4
4            1   neg   H1      1    0.5
5            1   neg   H2      1    0.6
6            1   pos   H1      1    0.7
7            1   pos   H2      1    0.8


## Several module calls: a tuple becomes `step`

A tuple or list of equal-shaped tensors is stacked on a new
`step` axis (one entry per forward / Captum call). Default names
for stacked 2-D pieces are `(step, observation, node)`.


In [9]:
step0 = torch.tensor(
    [
        [0.10, 0.20],
        [0.30, 0.40],
    ]
)
step1 = torch.tensor(
    [
        [0.50, 0.60],
        [0.70, 0.80],
    ]
)
by_step = k2.map_node_attributions(
    attributions=(step0, step1),
    spec=spec,
    layer=1,
)
print(
    "dims:",
    by_step.dims,
)
print(by_step.to_dataframe(name="score").reset_index())

dims: ('step', 'observation', 'node')
   step  observation node  layer  score
0     0            0   H1      1    0.1
1     0            0   H2      1    0.2
2     0            1   H1      1    0.3
3     0            1   H2      1    0.4
4     1            0   H1      1    0.5
5     1            0   H2      1    0.6
6     1            1   H1      1    0.7
7     1            1   H2      1    0.8


## Wrong width is an error

Passing hidden-layer scores as `layer=2` fails: `C` is one unit,
the tensor has two.


In [10]:
try:
    k2.map_node_attributions(
        attributions=scores_2x2,
        spec=spec,
        layer=2,
    )
except k2.Kpnn2Error as exc:
    print(exc)

Attribution tensor has the wrong number of units. Expected 1, got 2.


## What to map, and what not to

| Tensor you have | `layer=` |
|---|---|
| Input scores (`layer_dims[0]`) | `0` |
| Output of `MaskedLinear(spec.masks[i])` | `i + 1` |
| Output scores (`layer_dims[-1]`) | last layer |
| BatchNorm, dropout, unnamed module | do not map |

Last layer is `len(spec.layer_nodes) - 1`.

The function will happily label **any** tensor whose node-axis
length matches that layer. Matching width is not proof that the
scores came from those nodes. Only pass tensors whose units are
GraphSpec nodes.

For Captum on a trained feedforward net, see
[Getting started](../getting-started/) Step 6. For a cyclic
shared update, see the
[Recurrent example](../recurrent-example/).
